# Preparar estado inicial desde pp_reset

Genera y guarda los estados `sp_set` necesarios para que `Lanzar_simulacion_RRAM.ipynb`
pueda arrancar el ciclo directamente desde **PP_reset**.

```
init  →  generar_estados_sp_set  →  [ ejecutar con Lanzar_simulacion_RRAM.ipynb ]
```

### Flujo de uso
1. Ejecuta este notebook completo para generar `Init_data/phase_state_{n}_sp_set.*`
2. Abre `Lanzar_simulacion_RRAM.ipynb` y configura `start_from = 'pp_reset'` en la sección **EXEC**
3. Lanza las simulaciones desde allí

Logging: nivel global por env var `RRAM_LOG_LEVEL=DEBUG|INFO|WARNING`.

In [ ]:
%load_ext autoreload
%autoreload 2

import concurrent.futures
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

from scipy.signal import convolve2d
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap, BoundaryNorm
import numpy as np
import random
from RRAM import simulation_config, Percolation, utils
from RRAM.init_simulation import load_simulation_config
from RRAM.persistence import save_phase_state

ruta_raiz = Path.cwd()
print('Ruta raíz del proyecto:', ruta_raiz)


## 0 · Estructura de carpetas


In [ ]:
def setup_project_structure():
    for folder in ['Init_data', 'Results', 'Results/Figures', 'logs']:
        Path(folder).mkdir(parents=True, exist_ok=True)
        print(f'OK carpeta: {folder}/')

setup_project_structure()

carpeta_results = ruta_raiz / 'Results'
if carpeta_results.exists():
    shutil.rmtree(carpeta_results)
carpeta_results.mkdir(parents=True, exist_ok=True)
(carpeta_results / 'Figures').mkdir(parents=True, exist_ok=True)

log_dir = ruta_raiz / 'logs'
log_dir.mkdir(exist_ok=True)
for f in log_dir.iterdir():
    if f.is_file():
        try:
            f.unlink()
        except Exception as e:
            print(f'No se pudo eliminar {f}: {e}')

print('Estructura lista.')


## 1 · Generación de parámetros (CSV → Init_data)


In [ ]:
manager = simulation_config.ConfigManager()
carpeta_init = 'Init_data'

# ── Ajusta el barrido según tu experimento ───────────────────────────────
parametros_barrido = {
    # "num_filamentos": [1],
    # "grosor_filamento": [[30]],
    # "device_size_y": [40e-9,],
    "densidad_vacantes": [
        14,
        12,
        10,
        8,
        6,
        4,
        2,
        14,
        12,
        10,
        8,
        6,
        4,
        2,
        14,
        12,
        10,
        8,
        6,
        4,
        2,
    ],
    # "num_trampas": [20,20,20,20,20],
}
# ─────────────────────────────────────────────────────────────────────────

manager.add_zip(zip_params=parametros_barrido)
manager.export_to_init_data(carpeta_init)

num_simulaciones = len(manager.simulations)
num_filamentos = int(manager.simulations[0].params.get('num_filamentos', 2))
print(f"Generadas {num_simulaciones} configuraciones en {carpeta_init}/.")
print(f"num_filamentos por simulación: {num_filamentos}")


## 2 · INIT — pre-generar estados iniciales

Genera `Init_data/init_state_{i}.npz` para cada simulación del CSV.


In [ ]:
resultado = subprocess.run(
    [sys.executable, '-m', 'RRAM', 'init'],
    cwd=str(ruta_raiz),
    capture_output=True, text=True,
)
print(resultado.stdout)
if resultado.returncode != 0:
    print('STDERR:', resultado.stderr)
    raise RuntimeError('init falló')


## 3 · Generar estados sp_set

Para cada simulación construimos el dict `final_state_sp_set` que necesita `PP_reset`
y lo guardamos en `Init_data/phase_state_{n_save}_sp_set.*`.

El dispositivo se inicializa en **LRS** con una densidad determinada de vacantes assegurando el camino de percolación


In [ ]:
def crear_estado_inicial_garantizando_percolacion(
    eje_x: int,
    eje_y: int,
    num_filamentos: int,
    densidad_vacantes: float,
    peso_central: int = 70,
    grosores_filamento: list | int | None = None,
) -> np.ndarray:
    """
    Genera un estado inicial respetando ESTRICTAMENTE la densidad física.
    Si la densidad permite un camino serpenteante, lo crea.
    Si solo permite una fila, genera una línea recta.
    Si es insuficiente para cruzar, el filamento queda interrumpido.
    """
    estado = np.zeros((eje_x, eje_y), dtype=int)

    # 1. Parámetros físicos
    cte_red_nm = 0.25
    area_celda_nm2 = cte_red_nm**2

    # 2. Normalizar grosores
    if grosores_filamento is None:
        grosores = [2] * num_filamentos
    elif isinstance(grosores_filamento, (int, np.integer)):
        grosores = [int(grosores_filamento)] * num_filamentos
    else:
        grosores_raw = list(grosores_filamento)
        if len(grosores_raw) < num_filamentos:
            grosores_raw += [grosores_raw[-1]] * (num_filamentos - len(grosores_raw))
        grosores = [int(g) for g in grosores_raw[:num_filamentos]]

    ancho_zona = eje_x // num_filamentos

    for n in range(num_filamentos):
        inicio_rango = n * ancho_zona
        fin_rango = (n + 1) * ancho_zona - 1 if n < num_filamentos - 1 else eje_x - 1
        fila_central = (inicio_rango + fin_rango) // 2
        grosor = grosores[n]

        x_start = max(0, fila_central - grosor)
        x_end = min(eje_x, fila_central + grosor + 1)

        filas_region = x_end - x_start
        celdas_totales_region = filas_region * eje_y

        if celdas_totales_region <= 0:
            continue

        # 3. Calcular PRESUPUESTO ESTRICTO de vacantes
        area_region_nm2 = celdas_totales_region * area_celda_nm2
        num_vacantes_totales = int(round(area_region_nm2 * densidad_vacantes))
        num_vacantes_totales = min(num_vacantes_totales, celdas_totales_region)

        # ====================================================================
        # PASO A - CONSTRUIR EL CAMINO BASADO EN EL PRESUPUESTO
        # ====================================================================
        pos_x_actual = fila_central
        vacantes_restantes = num_vacantes_totales
        print(f"Filamento {n+1}/{num_filamentos}: Presupuesto de vacantes = {num_vacantes_totales}")
        for y in range(eje_y):
            # Si nos hemos quedado sin vacantes permitidas, paramos la construcción en seco
            if vacantes_restantes <= 0:
                break

            # A. Marcamos la posición actual en la columna Y
            if estado[pos_x_actual, y] == 0:
                estado[pos_x_actual, y] = 1
                vacantes_restantes -= 1
                
            # B. Si no es la última columna, decidimos el siguiente movimiento
            if y < eje_y - 1:
                # ¿Cuántas vacantes MÍNIMAS necesitamos para poder llegar al final en línea recta?
                pasos_minimos_restantes = (eje_y - 1) - y

                # SOLO nos curvamos (hacemos un codo) si tenemos vacantes de sobra
                if vacantes_restantes > pasos_minimos_restantes:
                    movimiento_x = random.choice([-1, 0, 1])
                    nuevo_x = pos_x_actual + movimiento_x
                    nuevo_x = max(x_start, min(nuevo_x, x_end - 1))

                    # Ejecutar el codo
                    if nuevo_x != pos_x_actual:
                        if estado[nuevo_x, y] == 0:
                            estado[nuevo_x, y] = 1
                            vacantes_restantes -= 1
                        pos_x_actual = nuevo_x

        # ====================================================================
        # PASO B - DISTRIBUIR LAS VACANTES SOBRANTES (si la densidad era alta)
        # ====================================================================
        if vacantes_restantes > 0:
            celdas_libres_x, celdas_libres_y = np.where(estado[x_start:x_end, :] == 0)
            celdas_libres_x += x_start

            num_libres = len(celdas_libres_x)
            vacantes_a_colocar = min(vacantes_restantes, num_libres)

            indices_elegidos = np.random.choice(num_libres, size=vacantes_a_colocar, replace=False)
            estado[celdas_libres_x[indices_elegidos], celdas_libres_y[indices_elegidos]] = 1

    return estado

## Representar estados iniciales pp_reset

In [ ]:
def representar_estado(estado: np.ndarray, titulo: str = "Estado Inicial - Distribución de Vacantes"):
    plt.figure(figsize=(10, 6))
    cmap = ListedColormap(["#F0F0F0", "#E63946"])
    imagen = plt.imshow(estado, cmap=cmap, aspect="equal", interpolation="none")
    cbar = plt.colorbar(imagen, ticks=[0, 1])
    cbar.ax.set_yticklabels(["Óxido", "Vacante"])
    plt.title(titulo, fontsize=14, pad=15)
    plt.xlabel("Eje X — distancia entre electrodos (columnas)", fontsize=12)
    plt.ylabel("Eje Y — alto del dispositivo (filas)", fontsize=12)
    plt.tight_layout()
    plt.show()
    plt.savefig(ruta_raiz / 'Results' / 'Figures' / 'estado_inicial_simple.png', dpi=300)


def representar_estado_con_zonas(
    estado: np.ndarray,
    num_filamentos: int,
    grosores_filamento=None,
    titulo: str = "Estado Inicial - Zonas de Generación de Vacantes",
):
    """
    Representa la matriz de estado con vacantes coloreadas según su número de
    vecinos vacantes directos (arriba/abajo/izq/der). Superpone un rectángulo
    que delimita la zona de cada filamento.

    Convención de ejes:
        eje_x = columnas = distancia entre electrodos (eje horizontal del plot)
        eje_y = filas    = alto del dispositivo      (eje vertical del plot)

    Colores de vacantes:
        azul oscuro → 0 vecinos (vacante aislada)
        verde       → 1 vecino
        amarillo    → 2 vecinos
        naranja     → 3 vecinos
        rojo        → 4 vecinos
    """
    # estado.shape = (filas, columnas) = (eje_y, eje_x)
    eje_y, eje_x = estado.shape

    # --- Calcular vecinos para todas las vacantes ---
    es_vacante = (estado == 1).astype(int)
    kernel = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]])
    conteo = convolve2d(es_vacante, kernel, mode="same", boundary="fill", fillvalue=0)

    # display: 0 = óxido, 1..5 = vacante con 0..4 vecinos
    display = np.zeros_like(estado, dtype=int)
    display[es_vacante == 1] = conteo[es_vacante == 1] + 1

    # --- Colormap discreto de 6 niveles ---
    colores = ["#E8E8E8", "#264653", "#2A9D8F", "#E9C46A", "#F4A261", "#E63946"]
    cmap = ListedColormap(colores)
    norm = BoundaryNorm(np.arange(-0.5, 6.5, 1), cmap.N)

    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(display, cmap=cmap, norm=norm, aspect="equal", interpolation="none")

    cbar = fig.colorbar(im, ax=ax, ticks=[0, 1, 2, 3, 4, 5])
    cbar.ax.set_yticklabels(["Óxido", "0 vec.", "1 vec.", "2 vec.", "3 vec.", "4 vec."])

    # --- Normalizar grosores ---
    if grosores_filamento is None:
        grosores = [2] * num_filamentos
    elif isinstance(grosores_filamento, (int, np.integer)):
        grosores = [int(grosores_filamento)] * num_filamentos
    else:
        grosores_raw = list(grosores_filamento)
        if len(grosores_raw) < num_filamentos:
            grosores_raw += [grosores_raw[-1]] * (num_filamentos - len(grosores_raw))
        grosores = [int(g) for g in grosores_raw[:num_filamentos]]

    # --- Rectángulo de zona de filamentos ---
    # Los filamentos se distribuyen a lo largo del eje_y (filas = alto del dispositivo)
    ancho_zona = eje_y // num_filamentos
    for n in range(num_filamentos):
        inicio_rango = n * ancho_zona
        fin_rango = (n + 1) * ancho_zona - 1 if n < num_filamentos - 1 else eje_y - 1
        fila_central = (inicio_rango + fin_rango) // 2
        grosor = grosores[n]

        fila_ini = max(0, fila_central - grosor)
        fila_fin = min(eje_y, fila_central + grosor + 1)

        rect = patches.Rectangle(
            (-0.5, fila_ini - 0.5),  # esquina superior-izquierda (col, fila)
            eje_x,                   # ancho = distancia entre electrodos (columnas)
            fila_fin - fila_ini,     # alto = filas de la zona
            linewidth=0.5,
            edgecolor="#457B9D",
            facecolor="none",
            alpha = 0.5,
            label="Zona permitida" if n == 0 else "",
        )
        ax.add_patch(rect)

    ax.legend(loc="upper right")
    ax.set_title(titulo, fontsize=14, pad=15)
    ax.set_xlabel("Eje X — distancia entre electrodos (columnas)", fontsize=12)
    ax.set_ylabel("Eje Y — alto del dispositivo (filas)", fontsize=12)
    plt.savefig(ruta_raiz / 'Results' / 'Figures' / 'estado_inicial_zonas.png', dpi=300)
    plt.tight_layout()
    plt.show()


In [ ]:
def representar_estado_combinado(
    estado: np.ndarray,
    num_filamentos: int,
    grosores_filamento=None,
    titulo_general: str = "Estado Inicial del Dispositivo RRAM",
    ruta_raiz: Path | str | None = None,
    nombre_archivo: str = "estado_inicial_combinado.png",
):
    """
    Crea una figura compuesta con dos paneles:
    1. Representación binaria simple (Óxido vs Vacantes).
    2. Representación avanzada con zonas delimitadas y colores por número de vecinos.
    """
    eje_y, eje_x = estado.shape

    # 1. Crear figura con 1 fila y 2 columnas
    # Ajustamos el tamaño para que quepan bien ambas gráficas
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8))

    # ==========================================
    # PANEL IZQUIERDO (ax1) - ESTADO SIMPLE
    # ==========================================
    cmap_simple = ListedColormap(["#F0F0F0", "#E63946"])
    im1 = ax1.imshow(estado, cmap=cmap_simple, aspect="equal", interpolation="none", origin="lower")

    # fraction y pad ayudan a que la barra de color tenga el mismo alto que la imagen
    cbar1 = fig.colorbar(im1, ax=ax1, ticks=[0, 1], fraction=0.046, pad=0.04)
    cbar1.ax.set_yticklabels(["Óxido", "Vacante"])

    ax1.set_title("Distribución Simple", fontsize=14, pad=15)
    ax1.set_xlabel("Eje X — distancia entre electrodos (columnas)", fontsize=12)
    ax1.set_ylabel("Eje Y — alto del dispositivo (filas)", fontsize=12)

    # ==========================================
    # PANEL DERECHO (ax2) - ZONAS Y VECINOS
    # ==========================================
    es_vacante = (estado == 1).astype(int)
    kernel = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]])
    conteo = convolve2d(es_vacante, kernel, mode="same", boundary="fill", fillvalue=0)

    display = np.zeros_like(estado, dtype=int)
    display[es_vacante == 1] = conteo[es_vacante == 1] + 1

    colores = ["#E8E8E8", "#264653", "#2A9D8F", "#E9C46A", "#F4A261", "#E63946"]
    cmap_zonas = ListedColormap(colores)
    norm = BoundaryNorm(np.arange(-0.5, 6.5, 1), cmap_zonas.N)

    im2 = ax2.imshow(display, cmap=cmap_zonas, norm=norm, aspect="equal", interpolation="none", origin="lower")
    cbar2 = fig.colorbar(im2, ax=ax2, ticks=[0, 1, 2, 3, 4, 5], fraction=0.046, pad=0.04)
    cbar2.ax.set_yticklabels(["Óxido", "0 vec.", "1 vec.", "2 vec.", "3 vec.", "4 vec."])

    # Normalizar grosores
    if grosores_filamento is None:
        grosores = [2] * num_filamentos
    elif isinstance(grosores_filamento, (int, np.integer)):
        grosores = [int(grosores_filamento)] * num_filamentos
    else:
        grosores_raw = list(grosores_filamento)
        if len(grosores_raw) < num_filamentos:
            grosores_raw += [grosores_raw[-1]] * (num_filamentos - len(grosores_raw))
        grosores = [int(g) for g in grosores_raw[:num_filamentos]]

    ancho_zona = eje_y // num_filamentos
    for n in range(num_filamentos):
        inicio_rango = n * ancho_zona
        fin_rango = (n + 1) * ancho_zona - 1 if n < num_filamentos - 1 else eje_y - 1
        fila_central = (inicio_rango + fin_rango) // 2
        grosor = grosores[n]

        fila_ini = max(0, fila_central - grosor)
        fila_fin = min(eje_y, fila_central + grosor + 1)

        rect = patches.Rectangle(
            (-0.5, fila_ini - 0.5),
            eje_x,
            fila_fin - fila_ini,
            linewidth=0.5,
            edgecolor="#457B9D",
            facecolor="none",
            alpha = 0.3,
            label="Zona permitida" if n == 0 else "",
        )
        ax2.add_patch(rect)

    ax2.legend(loc="upper right")
    ax2.set_title("Zonas y Vecindad", fontsize=14, pad=15)
    ax2.set_xlabel("Eje X — distancia entre electrodos (columnas)", fontsize=12)
    # Quitamos la etiqueta Y del segundo gráfico para no ser redundantes
    ax2.set_ylabel("", fontsize=12)

    # ==========================================
    # CONFIGURACIÓN GLOBAL Y GUARDADO
    # ==========================================
    # Título para toda la figura
    fig.suptitle(titulo_general, fontsize=16, fontweight="bold", y=1.02)

    plt.tight_layout()

    # Guardado de la imagen unificada
    if ruta_raiz is not None:
        ruta_raiz = Path(ruta_raiz)
        ruta_figuras = ruta_raiz / "Results" / "Figures"
        ruta_figuras.mkdir(parents=True, exist_ok=True)  # Crea las carpetas si no existen
        plt.savefig(ruta_figuras / nombre_archivo, dpi=300, bbox_inches="tight")

    # plt.show()


## Definir estado

In [ ]:
from RRAM.Generation import contar_vecinos_zona_filamento

def generar_estado_sp_set(
    num_simulation: int,
    init_data_dir: Path = Path("Init_data"),
    tiempo_sp_set: float = 0.0,
) -> bool:
    """
    Construye y guarda el estado final de SP_set para una simulación concreta.
    La densidad de vacantes se lee de cfg.params.densidad_vacantes.

    Returns:
        True si el dispositivo percola al final; False en caso contrario.
    """
    cfg      = load_simulation_config(num_simulation, init_data_dir=init_data_dir)
    params   = cfg.params
    sim_ctes = cfg.sim_ctes
    n_save   = num_simulation + 1

    eje_x = params.y_size   # filas  (alto del dispositivo)
    eje_y = params.x_size   # columnas (distancia entre electrodos)

    actual_state = crear_estado_inicial_garantizando_percolacion(
        eje_x=eje_x,
        eje_y=eje_y,
        num_filamentos=int(sim_ctes.num_filamentos),
        densidad_vacantes=params.densidad_vacantes,
        grosores_filamento=sim_ctes.grosor_filamento,
    )

    vecindad_inicial = contar_vecinos_zona_filamento(
        actual_state=actual_state,
        cf_ranges=cfg.cf_ranges,
    )

    percola = Percolation.is_path(actual_state.astype(int))

    Temperatura_final = np.full(
        (eje_x, eje_y + 2),
        fill_value=float(params.init_temp),
        dtype=np.float64,
    )

    final_state_sp_set = {
        "actual_state"      : actual_state,
        "sim_ctes"          : sim_ctes,
        "params"            : params,
        "Temperatura_final" : Temperatura_final,
        "percola"           : percola,
        "centros_calculados": list(cfg.cf_centros),
        "tiempo_sp_set"     : tiempo_sp_set,
        "vecindad_inicial"  : vecindad_inicial,
    }

    save_phase_state(
        state_dict=final_state_sp_set,
        phase_name="sp_set",
        num_simulation=n_save,
        init_data_dir=init_data_dir,
    )
    return percola


In [ ]:
init_data_dir = Path('Init_data')
errores_percola = []

for num_sim in range(num_simulaciones):
    percola = generar_estado_sp_set(
        num_sim,
        init_data_dir=init_data_dir,
    )
    estado = '✓ percola' if percola else '⚠ NO percola'
    print(f'  sim {num_sim:3d} (n_save={num_sim+1:3d}): {estado}')
    if not percola:
        errores_percola.append(num_sim)

print(f'\nEstados sp_set generados: {num_simulaciones}')
if errores_percola:
    print(f'⚠  Simulaciones sin percolación: {errores_percola}')
else:
    print('Todas las simulaciones percolan correctamente.')


In [ ]:
from RRAM.persistence import load_phase_state
from RRAM.Generation import contar_vecinos_zona_filamento

for num_sim in range(num_simulaciones):
    cfg          = load_simulation_config(num_sim, init_data_dir=init_data_dir)
    state_dict   = load_phase_state("sp_set", num_sim + 1, init_data_dir, cfg)
    actual_state = state_dict["actual_state"]

    # ── Estadísticas de vecindad ────────────────────────────────────────
    vecindad = contar_vecinos_zona_filamento(actual_state, cf_ranges=cfg.cf_ranges)
    total    = vecindad["total_vacantes_zona"]
    linea    = "  ".join(f"{i} vec: {vecindad[f'vecinos_{i}']}" for i in range(5))
    print(f"Sim {num_sim} · total vacantes zona: {total}  |  {linea}")

    # ── Plot combinado: estado simple (izq) + vecindad por colores (der) ─
    representar_estado_combinado(
        estado=actual_state,
        num_filamentos=int(cfg.sim_ctes.num_filamentos),
        grosores_filamento=cfg.sim_ctes.grosor_filamento,
        titulo_general=(f"Estado inicial PP_reset — sim {num_sim}  (densidad={cfg.params.densidad_vacantes} vac/nm²)"),
        nombre_archivo=f"estado_inicial_combinado_sim_{cfg.params.densidad_vacantes}_{num_sim}.png",
        ruta_raiz=ruta_raiz,
    )


In [ ]:
def calcular_densidad_minima(filas_zona_permitida: int, distancia_electrodos: int) -> float:
    """
    Calcula la densidad mínima necesaria (vacantes/nm^2) para poder formar
    un camino de percolación recto de un electrodo a otro.

    Argumentos:
        filas_zona_permitida (int): El número total de filas que abarca el grosor del filamento.
                                    (Si tu 'grosor' era 2, esto suele ser 2*2 + 1 = 5 filas).
        distancia_electrodos (int): El número de columnas de la matriz (eje_y).

    Retorna:
        float: La densidad mínima en vacantes/nm^2.
    """
    # 1. Recuperar la constante física del área de la celda
    cte_red_nm = 0.25
    area_celda_nm2 = cte_red_nm**2

    # 2. Calcular el área física total de la zona de generación
    area_total_nm2 = filas_zona_permitida * distancia_electrodos * area_celda_nm2
    print(f"Área total de la zona permitida: {area_total_nm2:.2f} nm²")

    # 3. El número mínimo de vacantes es una línea recta de lado a lado
    vacantes_minimas = distancia_electrodos

    # 4. Calcular la densidad (Densidad = Vacantes / Área)
    densidad_minima = vacantes_minimas / area_total_nm2

    return densidad_minima


# ==========================================
# Ejemplo de uso:
# ==========================================
DISTANCIA_X = 40  # eje_x distancia entre electrodos (columnas)
GROSOR_FILAS = 30 
filas_zona_permitida = 2 * GROSOR_FILAS + 1

densidad_min = calcular_densidad_minima(filas_zona_permitida, DISTANCIA_X)

print(f"Para una distancia de {DISTANCIA_X} celdas y un grosor de {GROSOR_FILAS} filas:")
print(f"Se necesitan mínimo {DISTANCIA_X} vacantes.")
print(f"La densidad mínima que debes introducir es: {densidad_min:.4f} vacantes/nm²")
